# Coverage-Guided Fuzz Test Input Generation (SuT: `calculate_shipping_fee`)

In this exercise, we'll implement a simple **coverage-guided fuzzing** algorithm that generates random test inputs and selects those that increase structural coverages.

In [2]:
import os, sys

# exercises → unit folder → modules → project_root
MODULES_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CODE_DIR = os.path.join(MODULES_ROOT, "exercise_artifacts", "code")
MODEL_DIR = os.path.join(MODULES_ROOT, "exercise_artifacts", "model")
DATA_DIR = os.path.join(MODULES_ROOT, "exercise_artifacts", "data")

if CODE_DIR not in sys.path:
    sys.path.append(CODE_DIR)

if MODEL_DIR not in sys.path:
    sys.path.append(MODEL_DIR)

if DATA_DIR not in sys.path:
    sys.path.append(DATA_DIR)

print("Added to sys.path:", CODE_DIR)
print("Added to sys.path:", MODEL_DIR)
print("Added to sys.path:", DATA_DIR)

from shipping_fee_instrumentation import *

stmt_tracker = StatementCoverageTracker()
branch_tracker = BranchCoverageTracker()
path_tracker = PathCoverageTracker()

print("Three structural coverage trackers have been initialized.")

Added to sys.path: /workspace/modules/exercise_artifacts/code
Added to sys.path: /workspace/modules/exercise_artifacts/model
Added to sys.path: /workspace/modules/exercise_artifacts/data
Three structural coverage trackers have been initialized.


## Random Input Generator

In [3]:
# Pure random generator (baseline)
import random

def generate_random_test_input():
    order_total = random.randint(0, 120000)
    weight_kg = round(random.uniform(0, 50), 2)
    distance_km = random.randint(0, 200)
    is_island = random.choice([True, False])
    membership = random.choice(["NONE", "WOW"])
    coupon_type = random.choice(["NONE", "NEW_USER"])
    return order_total, weight_kg, distance_km, is_island, membership, coupon_type

## Statement Coverage-Guided Generation

In [4]:
BUDGET = 100

test_inputs = []

for i in range(BUDGET):
    stmt_tracker.reset()

    # Get coverage of currently selected test inputs
    for selected_test_input in test_inputs:
        stmt_tracker.run_test(*selected_test_input)
    recent_coverage, _, _ = stmt_tracker.calculate_coverage()
    
    # Generate a new random test input
    random_test_input = generate_random_test_input()

    # Run the new test input and check if coverage increases
    stmt_tracker.run_test(*random_test_input)
    new_coverage, _, _ = stmt_tracker.calculate_coverage()
    
    # Select the test input if it increases coverage
    if new_coverage > recent_coverage:
        test_inputs.append(random_test_input)

# Print final report for selected test inputs
stmt_tracker.reset()
for test_input in test_inputs:
    stmt_tracker.run_test(*test_input)
stmt_tracker.print_report()

STATEMENT COVERAGE REPORT

Overall Coverage: 100.00% (12/12 items)
Total Tests: 4

Test Cases:
1: (56188, 4.03, 182, False, 'NONE', 'NONE')
2: (27196, 26.62, 43, True, 'WOW', 'NEW_USER')
3: (58155, 21.31, 3, True, 'WOW', 'NONE')
4: (61500, 17.33, 16, False, 'NONE', 'NONE')

| Test input             | 1    | 2    | 3    | 4    | Covered   |
|------------------------|------|------|------|------|-----------|
| fee = 0                | O    | O    | O    | O    | O         |
| fee += 3000            | X    | O    | X    | X    | O         |
| fee += 0 (weight)      | O    | X    | X    | X    | O         |
| fee += 2000            | X    | X    | X    | O    | O         |
| fee += 5000            | X    | O    | O    | X    | O         |
| fee += 0 (distance)    | X    | X    | O    | X    | O         |
| fee += 1000            | X    | O    | X    | O    | O         |
| fee += 3000 (distance) | O    | X    | X    | X    | O         |
| fee += 4000            | X    | O    | O    | X    | 

## Branch Coverage-Guided Generation

In [5]:
BUDGET = 100

test_inputs = []

for i in range(BUDGET):
    branch_tracker.reset()

    # Get coverage of currently selected test inputs
    for selected_test_input in test_inputs:
        branch_tracker.run_test(*selected_test_input)
    recent_coverage, _, _ = branch_tracker.calculate_coverage()
    
    # Generate a new random test input
    random_test_input = generate_random_test_input()

    # Run the new test input and check if coverage increases
    branch_tracker.run_test(*random_test_input)
    new_coverage, _, _ = branch_tracker.calculate_coverage()
    
    # Select the test input if it increases coverage
    if new_coverage > recent_coverage:
        test_inputs.append(random_test_input)

# Print final report for selected test inputs
branch_tracker.reset()
for test_input in test_inputs:
    branch_tracker.run_test(*test_input)
branch_tracker.print_report()

BRANCH COVERAGE REPORT

Overall Coverage: 100.00% (16/16 items)
Total Tests: 7

Test Cases:
1: (26828, 39.02, 63, False, 'NONE', 'NEW_USER')
2: (89412, 2.98, 182, False, 'NONE', 'NEW_USER')
3: (92356, 26.5, 113, False, 'WOW', 'NEW_USER')
4: (117352, 39.44, 9, False, 'NONE', 'NEW_USER')
5: (23314, 13.31, 71, True, 'NONE', 'NEW_USER')
6: (7587, 8.34, 95, False, 'NONE', 'NONE')
7: (32197, 9.48, 18, False, 'WOW', 'NONE')

| Test input                     | 1    | 2    | 3    | 4    | 5     | 6    | 7    | Covered   |
|--------------------------------|------|------|------|------|-------|------|------|-----------|
| order_total < 40000: True      | O    | X    | X    | X    | O     | O    | O    | O         |
| order_total < 40000: False     | X    | O    | O    | O    | X     | X    | X    | O         |
| weight_kg <= 5: True           | X    | O    | X    | X    | X     | X    | X    | O         |
| weight_kg <= 5: False          | O    | X    | O    | O    | O     | O    | O    | O       

## Path Coverage-Guided Generation

In [6]:
BUDGET = 100

test_inputs = []

for i in range(BUDGET):
    path_tracker.reset()

    # Get coverage of currently selected test inputs
    for selected_test_input in test_inputs:
        path_tracker.run_test(*selected_test_input)
    recent_coverage, _, _ = path_tracker.calculate_coverage()
    
    # Generate a new random test input
    random_test_input = generate_random_test_input()

    # Run the new test input and check if coverage increases
    path_tracker.run_test(*random_test_input)
    new_coverage, _, _ = path_tracker.calculate_coverage()
    
    # Select the test input if it increases coverage
    if new_coverage > recent_coverage:
        test_inputs.append(random_test_input)

# Print final report for selected test inputs
path_tracker.reset()
for test_input in test_inputs:
    path_tracker.run_test(*test_input)
path_tracker.print_report()

PATH COVERAGE REPORT

Overall Coverage: 38.19% (55/144 items)
Total Tests: 55

Test Cases:
1: (39247, 24.49, 178, False, 'WOW', 'NEW_USER')
2: (8510, 48.04, 70, True, 'NONE', 'NEW_USER')
3: (94626, 37.08, 145, False, 'NONE', 'NEW_USER')
4: (56985, 15.09, 13, True, 'NONE', 'NEW_USER')
5: (38926, 2.09, 111, False, 'WOW', 'NONE')
6: (85915, 1.5, 87, False, 'WOW', 'NEW_USER')
7: (83027, 30.92, 99, False, 'WOW', 'NONE')
8: (25915, 39.5, 113, False, 'NONE', 'NONE')
9: (115404, 37.14, 133, False, 'WOW', 'NEW_USER')
10: (112596, 23.28, 2, True, 'WOW', 'NONE')
11: (13495, 16.62, 194, False, 'NONE', 'NONE')
12: (26984, 7.55, 99, False, 'NONE', 'NEW_USER')
13: (2552, 3.03, 40, True, 'WOW', 'NONE')
14: (84517, 41.03, 194, True, 'WOW', 'NONE')
15: (4654, 43.85, 48, False, 'WOW', 'NONE')
16: (117773, 31.46, 96, False, 'NONE', 'NONE')
17: (112926, 24.32, 88, True, 'NONE', 'NEW_USER')
18: (38167, 13.93, 83, True, 'NONE', 'NEW_USER')
19: (113622, 32.34, 58, True, 'NONE', 'NONE')
20: (110397, 13.64, 90,

## 🤔 Think: Smarter Random Input Generator

Our `pure` random input generator struggles to achieve high coverage.

**Why?** Uniform random inputs across six parameters often miss edge combinations that trigger meaningful branches (e.g., threshold boundaries, island surcharge, membership/coupon interactions).

---

### ✅ Your Task

Modify the `generate_random_test_input()` above to include **stronger bias toward boundary values** for:
- `order_total` around `40000`
- `weight_kg` around `5` and `20`
- `distance_km` around `10` and `50`
- categorical combinations of `membership` / `coupon_type`

This simple biasing strategy can significantly improve structural coverage.

---

## 🎯 Smarter Random Generators in Fuzzing

The ultimate goal of fuzzing is not just to generate random inputs —  
but to generate **smarter random inputs**.

Below are representative approaches used in practice:

| Approach                         | Required SuT Knowledge                     | Example                          |
|----------------------------------|--------------------------------------------|----------------------------------|
| Pure Random                      | 🟢 None (Black-box)                         | Uniform random sampling          |
| Biased Random                    | 🟢 Very Low (general heuristics only)       | Small-int bias, boundary values  |
| Mutation-Based                   | 🟡 Low (seed inputs required)               | AFL-style bit/byte mutations     |
| Evolutionary / Genetic           | 🟡 Grey-box (fitness such as coverage)      | Coverage-based genetic fuzzing   |
| Grammar-Based                    | 🟠 Input format / grammar knowledge         | JSON/XML grammar fuzzing         |
| Constraint-Based / Symbolic      | 🔴 High (White-box, path constraint access) | KLEE, symbolic execution engines |


---

### 🔎 Reflection

- Why do boundary-focused inputs improve coverage in this case?
- What kind of bias would help if the input domain were images? Strings? Network packets?
- How much knowledge about the system under test (SuT) should a fuzzing strategy assume?

Keep in mind:

> Pure random is rarely efficient in practice.  
> **The key to effective fuzzing is choosing (or designing) a strategy that fits your SuT.**